In [1]:
!pip install -q torchmetrics seaborn opencv-python-headless pandas

In [2]:
import os
os.makedirs("/kaggle/working/evaluation/results", exist_ok=True)
os.makedirs("/kaggle/working/evaluation/figures", exist_ok=True)
os.makedirs("/kaggle/working/evaluation/fp_analysis/crops", exist_ok=True)

In [3]:
# Imports
import json, time, shutil
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from torchvision.ops import box_iou
from torchmetrics.detection.mean_ap import MeanAveragePrecision
import torchvision.transforms as T


# config
@dataclass
class CFG:
    ckpt: str = "/kaggle/input/datasets/c3m3nt/cv-project-checkpoints/faster_rcnn_best.pth"
    test_images_dir: str = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/images"
    test_labels_dir: str = "/kaggle/input/datasets/linxel/dfire-prepared/data/test/labels"

    # inference
    batch_size: int = 2
    num_workers: int = 2
    input_size: int = 640
    num_classes: int = 3                       # background + smoke + fire

    # thresholds
    score_low: float = 0.05                    # mAP
    score_prf: float = 0.50                    # P/R/F1, confusion
    iou_thresh: float = 0.50

    # FP analysis
    run_fp_analysis: bool = True
    fp_topk: int = 100

    out_dir: str = "/kaggle/working/evaluation"

CFG = CFG()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}  "
          f"({round(torch.cuda.get_device_properties(0).total_memory/1e9,1)} GB)")

for p in [CFG.ckpt, CFG.test_images_dir, CFG.test_labels_dir]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Not found: {p}")
n_imgs = len([f for f in os.listdir(CFG.test_images_dir)
              if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp"))])
print(f"Test images on disk: {n_imgs}")


for p in [CFG.ckpt, CFG.test_images_dir, CFG.test_labels_dir]:
    if not os.path.exists(p):
        raise FileNotFoundError(f"Not found: {p}")
n_imgs = len([f for f in os.listdir(CFG.test_images_dir)
              if f.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp"))])
print(f"Test images on disk: {n_imgs}")

# Dataset
CLASS_NAMES = {1: "smoke", 2: "fire"}   # torchvision: 0 = background

class FireSmokeDataset(torch.utils.data.Dataset):
    IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    def __init__(self, images_dir, labels_dir, transforms=None):
        self.images_dir = Path(images_dir)
        self.labels_dir = Path(labels_dir)
        self.transforms = transforms
        self.files = sorted(p for p in self.images_dir.iterdir()
                            if p.suffix.lower() in self.IMG_EXTS)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_path = self.files[idx]
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        boxes, labels = [], []
        lbl = self.labels_dir / (img_path.stem + ".txt")
        if lbl.exists():
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls, xc, yc, bw, bh = map(float, parts)
                x1 = max(0.0, (xc - bw/2) * w); y1 = max(0.0, (yc - bh/2) * h)
                x2 = min(w, (xc + bw/2) * w); y2 = min(h, (yc + bh/2) * h)
                if x2 <= x1 or y2 <= y1:
                    continue
                boxes.append([x1, y1, x2, y2])
                labels.append(1 if int(cls) == 0 else 2)
        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        if self.transforms:
            img = self.transforms(img)
        target = {
            "boxes": boxes, "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (boxes[:,3]-boxes[:,1])*(boxes[:,2]-boxes[:,0]) if boxes.numel()
                    else torch.zeros((0,), dtype=torch.float32),
            "iscrowd": torch.zeros((boxes.shape[0],), dtype=torch.int64),
        }
        return img, target, str(img_path)

def collate_fn(batch):
    return tuple(zip(*batch))

test_ds = FireSmokeDataset(CFG.test_images_dir, CFG.test_labels_dir,
                           transforms=T.ToTensor())
print(f"Test dataset: {len(test_ds)} images")
test_loader = torch.utils.data.DataLoader(
    test_ds, batch_size=CFG.batch_size, shuffle=False,
    num_workers=CFG.num_workers, collate_fn=collate_fn, pin_memory=True,
)


# Model
def load_faster_rcnn(ckpt_path, num_classes, input_size, device):
    from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
    from torchvision.models.detection import fasterrcnn_resnet50_fpn
    model = fasterrcnn_resnet50_fpn(weights=None)
    in_f = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_f, num_classes)
    model.transform.min_size = (input_size,)
    model.transform.max_size = input_size
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"])
    model.to(device).eval()
    return model, ckpt.get("epoch", "?")

model, ckpt_epoch = load_faster_rcnn(CFG.ckpt, CFG.num_classes,
                                     CFG.input_size, DEVICE)
print(f"Loaded checkpoint (epoch {ckpt_epoch})")



# Inference
@torch.no_grad()
def run_inference(model, loader, device, score_thresh):
    preds, targets, records = [], [], []
    n_img = 0
    t0 = time.time()
    for images, tgts, paths in tqdm(loader, desc="Inference"):
        images = [im.to(device, non_blocking=True) for im in images]
        outputs = model(images)
        for out, tgt, p in zip(outputs, tgts, paths):
            keep = out["scores"] >= score_thresh
            preds.append({
                "boxes": out["boxes"][keep].cpu(),
                "scores": out["scores"][keep].cpu(),
                "labels": out["labels"][keep].cpu(),
            })
            targets.append({
                "boxes": tgt["boxes"].cpu(),
                "labels": tgt["labels"].cpu(),
            })
            records.append({
                "path": p,
                "boxes": out["boxes"][keep].cpu().numpy(),
                "scores": out["scores"][keep].cpu().numpy(),
                "labels": out["labels"][keep].cpu().numpy(),
                "gt_boxes": tgt["boxes"].cpu().numpy(),
                "gt_labels": tgt["labels"].cpu().numpy(),
            })
        n_img += len(images)
    fps = n_img / max(time.time() - t0, 1e-6)
    return preds, targets, records, fps

preds, targets, records, fps = run_inference(model, test_loader, DEVICE,
                                             CFG.score_low)
print(f"Inference FPS: {fps:.2f}")



# mAP
def compute_map(preds, targets):
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox",
                                  class_metrics=True)
    p_remap = [{"boxes": p["boxes"], "scores": p["scores"],
                "labels": p["labels"] - 1} for p in preds]
    t_remap = [{"boxes": t["boxes"], "labels": t["labels"] - 1} for t in targets]
    metric.update(p_remap, t_remap)
    return metric.compute()

m = compute_map(preds, targets)
mAP50   = float(m["map_50"])
mAP5095 = float(m["map"])
per_cls = [float(x) for x in m["map_per_class"]]
print(f"mAP@0.5      = {mAP50:.4f}")
print(f"mAP@0.5:0.95 = {mAP5095:.4f}")
print(f"mAP@0.5 [smoke, fire] = {per_cls}")



# Per-class P/R/F1
def per_class_prf(preds, targets, iou_thresh, score_thresh):
    stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    for p, t in zip(preds, targets):
        for cls in [1, 2]:
            pm = (p["labels"] == cls) & (p["scores"] >= score_thresh)
            pb = p["boxes"][pm]
            tb = t["boxes"][t["labels"] == cls]
            if len(tb) == 0:
                stats[cls]["fp"] += len(pb); continue
            if len(pb) == 0:
                stats[cls]["fn"] += len(tb); continue
            ious = box_iou(pb, tb)
            matched = set()
            for i in range(len(pb)):
                bi, bj = 0.0, -1
                for j in range(len(tb)):
                    if j in matched: continue
                    v = ious[i, j].item()
                    if v > bi: bi, bj = v, j
                if bi >= iou_thresh and bj >= 0:
                    stats[cls]["tp"] += 1; matched.add(bj)
                else:
                    stats[cls]["fp"] += 1
            stats[cls]["fn"] += len(tb) - len(matched)
    out = {}
    for cls, s in stats.items():
        tp, fp, fn = s["tp"], s["fp"], s["fn"]
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        out[CLASS_NAMES[cls]] = {"precision": round(prec,4),
                                 "recall":    round(rec,4),
                                 "f1":        round(f1,4),
                                 "tp":tp, "fp":fp, "fn":fn}
    return out

prf = per_class_prf(preds, targets, CFG.iou_thresh, CFG.score_prf)
print("Per-class P/R/F1:", json.dumps(prf, indent=2))



# Confusion matrix
def confusion_matrix_counts(preds, targets, iou_thresh, score_thresh):
    cm = np.zeros((3, 3), dtype=int)
    for p, t in zip(preds, targets):
        matched = set()
        for i in range(len(p["boxes"])):
            if p["scores"][i] < score_thresh: continue
            pc = int(p["labels"][i])
            bi, bj = 0.0, -1
            for j in range(len(t["boxes"])):
                if j in matched: continue
                v = box_iou(p["boxes"][i:i+1], t["boxes"][j:j+1]).item()
                if v > bi: bi, bj = v, j
            if bi >= iou_thresh and bj >= 0:
                cm[int(t["labels"][bj]), pc] += 1; matched.add(bj)
            else:
                cm[0, pc] += 1
        for j in range(len(t["boxes"])):
            if j not in matched:
                cm[int(t["labels"][j]), 0] += 1
    return cm

cm = confusion_matrix_counts(preds, targets, CFG.iou_thresh, CFG.score_prf)
print("Confusion matrix (rows=GT bg/smoke/fire, cols=Pred bg/smoke/fire):")
print(cm)



# Figures
fig_dir = Path(CFG.out_dir) / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["bg","smoke","fire"],
            yticklabels=["bg","smoke","fire"])
plt.xlabel("Predicted"); plt.ylabel("Ground truth")
plt.title(f"Confusion matrix @ IoU={CFG.iou_thresh}, score={CFG.score_prf}")
plt.tight_layout()
plt.savefig(fig_dir / "confusion_matrix.png", dpi=150); plt.close()

cls_names = list(prf.keys())
fig, ax = plt.subplots(figsize=(6, 4))
x = np.arange(len(cls_names)); w = 0.25
for i, metric in enumerate(["precision","recall","f1"]):
    ax.bar(x + (i-1)*w, [prf[c][metric] for c in cls_names], w, label=metric)
ax.set_xticks(x); ax.set_xticklabels(cls_names)
ax.set_ylim(0, 1); ax.set_ylabel("Score"); ax.legend()
ax.set_title(f"Per-class P/R/F1 @ IoU={CFG.iou_thresh}, score={CFG.score_prf}")
plt.tight_layout()
plt.savefig(fig_dir / "per_class_prf.png", dpi=150); plt.close()
print(f"Figures → {fig_dir}")



# FP extraction
df_fp = None
if CFG.run_fp_analysis:
    import cv2, pandas as pd
    fp_dir = Path(CFG.out_dir) / "fp_analysis"
    crops_dir = fp_dir / "crops"
    crops_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for rec in records:
        for i in range(len(rec["boxes"])):
            if rec["scores"][i] < CFG.score_prf: continue
            pb = rec["boxes"][i]
            matched = False
            if len(rec["gt_boxes"]) > 0:
                ious = box_iou(torch.tensor(pb[None]),
                               torch.tensor(rec["gt_boxes"]))
                if ious.max().item() >= CFG.iou_thresh:
                    matched = True
            if matched: continue
            x1, y1, x2, y2 = map(int, pb)
            img = cv2.imread(rec["path"])
            if img is None: continue
            h, w = img.shape[:2]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(w, x2), min(h, y2)
            crop = img[y1:y2, x1:x2]
            if crop.size == 0: continue
            fname = f"{Path(rec['path']).stem}_fp{i}.jpg"
            cv2.imwrite(str(crops_dir / fname), crop)
            rows.append({
                "image": rec["path"], "crop": fname,
                "pred_class": CLASS_NAMES.get(int(rec["labels"][i]), "?"),
                "score": float(rec["scores"][i]),
                "category": "",   # clouds / fog / lighting / bright
                "notes": "",
            })
    df_fp = pd.DataFrame(rows).sort_values("score", ascending=False).head(CFG.fp_topk)
    df_fp.to_csv(fp_dir / "fp_candidates.csv", index=False)
    print(f"Saved {len(df_fp)} FP candidates → {fp_dir/'fp_candidates.csv'}")



# Save metrics JSON
results = {
    "model": "fasterrcnn_resnet50_fpn",
    "checkpoint": CFG.ckpt,
    "checkpoint_epoch": ckpt_epoch,
    "split": "test",
    "test_images_dir": CFG.test_images_dir,
    "num_images": len(test_ds),
    "mAP@0.5": mAP50,
    "mAP@0.5:0.95": mAP5095,
    "per_class_mAP@0.5": {
        "smoke": per_cls[0] if len(per_cls) > 0 else None,
        "fire":  per_cls[1] if len(per_cls) > 1 else None,
    },
    "per_class_PRF@0.5": prf,
    "confusion_matrix": cm.tolist(),
    "fps": fps,
    "iou_thresh": CFG.iou_thresh,
    "score_thresh_prf": CFG.score_prf,
    "score_thresh_inference": CFG.score_low,
}
res_path = Path(CFG.out_dir) / "results" / "faster_rcnn_test.json"
res_path.parent.mkdir(parents=True, exist_ok=True)
res_path.write_text(json.dumps(results, indent=2))
print(f"Metrics JSON → {res_path}")



# Summary
print("\n" + "="*60)
print("FINAL SUMMARY — Faster R-CNN ResNet-50 FPN")
print("="*60)
print(f"Checkpoint epoch : {ckpt_epoch}")
print(f"Test images      : {len(test_ds)}")
print(f"mAP@0.5          : {mAP50:.4f}")
print(f"mAP@0.5:0.95     : {mAP5095:.4f}")
for c in ["smoke", "fire"]:
    p = prf[c]
    print(f"  {c:6s} P/R/F1   : {p['precision']:.3f} / "
          f"{p['recall']:.3f} / {p['f1']:.3f}   "
          f"(tp={p['tp']}, fp={p['fp']}, fn={p['fn']})")
print(f"FPS (T4)         : {fps:.2f}")
if df_fp is not None:
    print(f"FP candidates    : {len(df_fp)}")
print("="*60)
print(f"Artifacts → {CFG.out_dir}")


# Package artifacts
archive_path = "/kaggle/working/evaluation_output"
shutil.make_archive(archive_path, "zip", CFG.out_dir)
print(f"ZIP → {archive_path}.zip")

Device: cuda
GPU: Tesla T4  (15.6 GB)
Test images on disk: 4306
Test images on disk: 4306
Test dataset: 4306 images
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 249MB/s]


Loaded checkpoint (epoch 5)


Inference:   0%|          | 0/2153 [00:00<?, ?it/s]

Inference FPS: 20.81
mAP@0.5      = 0.6965
mAP@0.5:0.95 = 0.3551
mAP@0.5 [smoke, fire] = [0.3921356797218323, 0.3181488513946533]
Per-class P/R/F1: {
  "smoke": {
    "precision": 0.5139,
    "recall": 0.8037,
    "f1": 0.6269,
    "tp": 1851,
    "fp": 1751,
    "fn": 452
  },
  "fire": {
    "precision": 0.5964,
    "recall": 0.6855,
    "f1": 0.6379,
    "tp": 1973,
    "fp": 1335,
    "fn": 905
  }
}
Confusion matrix (rows=GT bg/smoke/fire, cols=Pred bg/smoke/fire):
[[   0 1741 1325]
 [ 443 1850   10]
 [ 894   11 1973]]
Figures → /kaggle/working/evaluation/figures
Saved 100 FP candidates → /kaggle/working/evaluation/fp_analysis/fp_candidates.csv
Metrics JSON → /kaggle/working/evaluation/results/faster_rcnn_test.json

FINAL SUMMARY — Faster R-CNN ResNet-50 FPN
Checkpoint epoch : 5
Test images      : 4306
mAP@0.5          : 0.6965
mAP@0.5:0.95     : 0.3551
  smoke  P/R/F1   : 0.514 / 0.804 / 0.627   (tp=1851, fp=1751, fn=452)
  fire   P/R/F1   : 0.596 / 0.685 / 0.638   (tp=1973, fp=1